# Walk Exploration

Interactive notebook for exploring random walks and embedding behavior on synthetic and real hypergraphs.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hypergraphx import Hypergraph

from src.utils import hmrw_embeddings

In [ ]:
# TODO: load or generate a hypergraph, then explore walk behavior.
# Example placeholders:
# H = Hypergraph(...)
# X = hmrw_embeddings(H, p=1.0, q=1.0, r=1.0, num_walks=10, walk_length=40, dim=64)
# print(X.shape)

In [ ]:
# Simple HMRW-style walk sampler (one or more walks per node)
walk_length = 40
walks_per_node = 1   # tune this
p, q, r = 1.0, 1.0, 1.0

nodes = sorted(H.get_nodes())
idx = {u: i for i, u in enumerate(nodes)}
edges = [frozenset(e) for e in H.get_edges()]

n = len(nodes)
d = np.zeros(n)
inc_edges = [[] for _ in range(n)]
for e_i, e in enumerate(edges):
    for u in e:
        i = idx[u]
        d[i] += 1
        inc_edges[i].append(e_i)

rng = np.random.default_rng(0)
walks = []

for _ in range(walks_per_node):
    for start in range(n):
        v_i = start
        walk = [v_i]
        prev_u_i, prev_e_i = None, None

        for _ in range(walk_length - 1):
            cand_e = inc_edges[v_i]
            raw_e = np.zeros(len(cand_e))
            for k, e2_i in enumerate(cand_e):
                e2 = edges[e2_i]
                if prev_e_i is None:
                    a_edge = 1.0
                elif e2_i == prev_e_i:
                    a_edge = 1.0 / p
                elif nodes[prev_u_i] in e2:
                    a_edge = 1.0
                else:
                    a_edge = 1.0 / q
                raw_e[k] = a_edge / len(e2)

            next_e_i = cand_e[int(rng.choice(len(cand_e), p=raw_e / raw_e.sum()))]
            e2 = edges[next_e_i]
            cand_x = [idx[x] for x in e2]

            raw_x = np.zeros(len(cand_x))
            prev_e_nodes = edges[prev_e_i] if prev_e_i is not None else None
            for k, x_i in enumerate(cand_x):
                if prev_u_i is None:
                    a_node = 1.0
                elif nodes[x_i] in prev_e_nodes:
                    a_node = 1.0
                else:
                    a_node = 1.0 / r
                raw_x[k] = a_node / d[x_i]

            x_i = cand_x[int(rng.choice(len(cand_x), p=raw_x / raw_x.sum()))]
            prev_u_i, prev_e_i, v_i = v_i, next_e_i, x_i
            walk.append(v_i)

        walks.append(walk)

print("number of walks:", len(walks), "walk length:", len(walks[0]))

In [ ]:
# Plot: number of newly discovered unique nodes at each step
new_nodes_per_step = np.zeros(walk_length)

for w in walks:
    seen = set()
    for t, v in enumerate(w):
        if v not in seen:
            new_nodes_per_step[t] += 1
            seen.add(v)

new_nodes_per_step /= len(walks)

plt.figure(figsize=(6, 4))
plt.plot(np.arange(1, walk_length + 1), new_nodes_per_step, marker="o", ms=3)
plt.xlabel("Walk step")
plt.ylabel("New unique nodes visited")
plt.title(f"HMRW exploration (walks_per_node={walks_per_node})")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()